# Implementasi Bab 5 — Decision Tree dan Pruning

Perbandingan pohon penuh, pre-pruning, dan cost-complexity post-pruning pada Breast Cancer Wisconsin [ML-MODUL-01, hlm. 39–40]. Pemilihan alpha latihan memakai validation set, bukan test set, sebagai adaptasi anti-leakage.


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
data = load_breast_cancer(as_frame=True)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=SEED, stratify=data.target
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=SEED, stratify=y_train_full
)
print("train/val/test:", X_train.shape, X_val.shape, X_test.shape)


In [ ]:
tree_full = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train)
tree_pruned = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=5, random_state=SEED
).fit(X_train, y_train)

pruning_path = tree_full.cost_complexity_pruning_path(X_train, y_train)
candidate_alphas = pruning_path.ccp_alphas[:-1]
validation_scores = []
for alpha in candidate_alphas:
    candidate = DecisionTreeClassifier(ccp_alpha=alpha, random_state=SEED)
    candidate.fit(X_train, y_train)
    validation_scores.append(accuracy_score(y_val, candidate.predict(X_val)))
selected_alpha = candidate_alphas[int(np.argmax(validation_scores))]
tree_ccp = DecisionTreeClassifier(ccp_alpha=selected_alpha, random_state=SEED)
tree_ccp.fit(X_train_full, y_train_full)
print("ccp_alpha terpilih:", selected_alpha)


In [ ]:
comparison = []
for name, candidate in {
    "Penuh": tree_full,
    "Pre-pruning": tree_pruned,
    "Post-pruning": tree_ccp,
}.items():
    train_X = X_train_full if name == "Post-pruning" else X_train
    train_y = y_train_full if name == "Post-pruning" else y_train
    train_accuracy = accuracy_score(train_y, candidate.predict(train_X))
    test_accuracy = accuracy_score(y_test, candidate.predict(X_test))
    comparison.append((name, train_accuracy, test_accuracy, train_accuracy - test_accuracy))
display(pd.DataFrame(comparison, columns=["model", "train", "test", "gap"]))


In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    tree_pruned,
    feature_names=data.feature_names,
    class_names=data.target_names,
    filled=True,
    rounded=True,
    fontsize=7,
)
plt.title("Decision Tree Pre-Pruning")
plt.show()


## Latihan 1: kombinasi pre-pruning

Grid kecil berikut memperlihatkan pengaruh `max_depth` dan `min_samples_leaf` [ML-MODUL-01, hlm. 40]. Kombinasi dipilih memakai validation set, lalu test set hanya dibuka sekali untuk evaluasi final.


In [ ]:
pre_pruning_results = []
for depth in [2, 4, 6, None]:
    for min_leaf in [1, 5, 10]:
        candidate = DecisionTreeClassifier(
            max_depth=depth, min_samples_leaf=min_leaf, random_state=SEED
        ).fit(X_train, y_train)
        validation_accuracy = accuracy_score(y_val, candidate.predict(X_val))
        pre_pruning_results.append((depth, min_leaf, validation_accuracy))
pre_pruning_frame = pd.DataFrame(
    pre_pruning_results, columns=["max_depth", "min_leaf", "validation_accuracy"]
)
display(pre_pruning_frame.sort_values("validation_accuracy", ascending=False))
best_row = pre_pruning_frame.sort_values("validation_accuracy", ascending=False).iloc[0]
selected_pre_pruned_tree = DecisionTreeClassifier(
    max_depth=None if pd.isna(best_row["max_depth"]) else int(best_row["max_depth"]),
    min_samples_leaf=int(best_row["min_leaf"]),
    random_state=SEED,
).fit(X_train_full, y_train_full)
final_test_accuracy = accuracy_score(y_test, selected_pre_pruned_tree.predict(X_test))
print(f"Akurasi test final: {final_test_accuracy:.3f}")


In [ ]:
plt.plot(candidate_alphas, validation_scores, marker="o")
plt.axvline(selected_alpha, color="red", linestyle="--", label="alpha terpilih")
plt.xlabel("ccp_alpha")
plt.ylabel("Akurasi validation")
plt.title("Cost Complexity Pruning Path")
plt.legend()
plt.show()


## Kesimpulan

Pohon penuh mudah menghafal training set. Pruning menukar sedikit akurasi training untuk gap generalisasi yang lebih kecil; parameter sebaiknya dipilih dari validation/CV, bukan test set [ML-MODUL-01, hlm. 40].
